# Defenders Analysis System Documentation

## Overview
The Defenders Analysis System is a comprehensive Python-based analytics tool designed to evaluate soccer/football defenders' performance across multiple competitions and seasons using StatsBomb data. The system calculates and normalizes various defensive metrics to provide detailed insights into defender capabilities.

## Core Components

### 1. Competition Configuration
`CompetitionConfig` class manages competition-specific settings including:
- Competition and season ID handling
- Calculation of minimum minutes threshold based on competition length
- Dynamic scaling of requirements based on competition size

### 2. Performance Metrics

#### 2.1 Box Defending (`calculate_box_defending`)
Evaluates defender's performance in the penalty box area:
- Tracks clearances, blocks, interceptions, and duels
- Uses weighted scoring system (0-1 scale)
- Considers spatial position of defensive actions
- Weights: Clearance (0.8), Block (1.0), Interception (1.2), Duel (1.0)

#### 2.2 Aerial Ability (`calculate_aerial_ability`)
Assesses defender's proficiency in aerial duels:
- Analyzes aerial duel outcomes
- Implements zone-based weighting system
- Weights based on field position:
  - Defensive third: 1.2x
  - Middle third: 1.0x
  - Attacking third: 0.8x

#### 2.3 One-v-One Defense (`calculate_one_v_one`)
Measures defender's ability in one-on-one situations:
- Evaluates tackle success rate
- Incorporates ball recovery metrics
- Considers pressure events
- Uses pressure factor multiplier (up to 1.2x)

#### 2.4 Defensive Pressure (`calculate_defensive_pressure`)
Analyzes pressing effectiveness:
- Tracks successful pressure events
- Evaluates pressure outcomes within 3 subsequent events
- Applies location-based weighting
- Considers team possession regains

#### 2.5 Ball Progression (`calculate_ball_progression`)
Measures defender's contribution to build-up play:
- Analyzes progressive passing
- Considers pass completion rates
- Zone-based progression values
- Weighted by field position and progression distance

### 3. Data Processing

#### 3.1 Normalization (`normalize_metric`)
Sophisticated normalization process including:
- Reliability factor based on minutes played
- Dynamic range adjustment
- Confidence interval calculation
- Outlier handling
- Weighted averaging based on playing time

#### 3.2 Main Analysis Function (`analyze_defender_performance`)
Comprehensive analysis pipeline:
- Data collection for each match
- Metric calculation
- Statistical aggregation
- Composite score generation
- Confidence interval calculation

### 4. Data Collection System

#### 4.1 League Configuration
Supports multiple competitions:
```python
LEAGUES = [
    {"competition_id": 9, "season_ids": [281]},  # Example league
    {"competition_id": 43, "season_ids": [106]}, # Another league
    # ... additional leagues
]
```

#### 4.2 Network Handling
Robust data fetching system:
- Retry mechanism with exponential backoff
- Session management
- Error handling
- Rate limiting protection

### 5. Output and Results

#### 5.1 Data Export
- Detailed CSV export with all metrics
- Summary report of top performers
- Competition-specific analysis
- Time-stamped output files

#### 5.2 Key Metrics in Output
- Composite score
- Individual metric scores (0-10 scale)
- Confidence intervals
- Minutes played
- Matches played
- Full match equivalents

## Usage

### Basic Implementation
```python
# Initialize configuration
config = CompetitionConfig(competition_id=9, season_id=281)

# Run analysis
results = analyze_defender_performance(
    competition_id=9,
    season_id=281
)
```

### Full Analysis Pipeline
```python
# Run complete analysis across all configured leagues
if __name__ == "__main__":
    main()
```

## Technical Requirements

### Dependencies
- pandas
- numpy
- statsbombpy
- requests
- tqdm
- scipy

### Data Requirements
- StatsBomb data access
- Minimum event data requirements:
  - Player position information
  - Event locations
  - Match timing data
  - Team identification

## Performance Considerations

### Minimum Thresholds
- Box defending: 10 events minimum
- Aerial ability: 15 duels minimum
- One-v-one: 10 tackles minimum
- Defensive pressure: 15 pressure events minimum
- Ball progression: 20 passes minimum

### Data Reliability
- Weighted averaging based on minutes played
- Confidence interval calculation
- Dynamic range adjustment based on sample size
- Outlier handling through robust statistical methods

## Limitations and Considerations
1. Requires consistent event data quality
2. Performance varies with data sample size
3. Dependent on StatsBomb API availability
4. Rate limiting considerations
5. Processing time scales with competition size

## Error Handling
- Robust error catching for API failures
- Data validation at each processing step
- Graceful degradation for missing data
- Detailed error logging

In [1]:




from tqdm import tqdm
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd
import numpy as np
from typing import List, Dict, Union, Tuple
from statsbombpy import sb
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import warnings
import glob
import os
warnings.simplefilter("ignore")
from scipy import stats  



class CompetitionConfig:
    def __init__(self, competition_id: int, season_id: int):
        """
        Initialize competition configuration with improved minutes threshold calculation
        """
        self.competition_id = competition_id
        self.season_id = season_id
        matches = sb.matches(competition_id=self.competition_id, season_id=self.season_id)
        self.total_matches = len(matches)
    
    def get_minimum_minutes(self) -> int:
        """
        Calculate minimum minutes threshold with more granular scaling
        """
        total_possible_minutes = self.total_matches * 90
        if self.total_matches <= 7:
            return 180  # At least 2 full matches for very short competitions
        elif self.total_matches <= 15:
            return total_possible_minutes * 0.15
        else:
            return total_possible_minutes * 0.10

In [2]:
pd.set_option('display.max_rows', None)

In [3]:
def calculate_box_defending(events: pd.DataFrame, defender: str, team: str) -> float:
    """
    Calculate box defending performance with improved normalization and weighting
    Returns a score between 0-1
    """
    defender_events = events[events['player'] == defender].copy()
    
    def is_in_box(location: List[float]) -> bool:
        if not isinstance(location, list) or len(location) != 2:
            return False
        x, y = location
        return x >= 75 and 15 <= y <= 65
    
    defender_events.loc[:, 'location'] = defender_events['location'].apply(
        lambda x: eval(x) if isinstance(x, str) else x
    )
    
    box_events = defender_events[defender_events['location'].apply(is_in_box)]
    defensive_types = ['Clearance', 'Block', 'Interception', 'Duel']
    defensive_attempts = box_events[box_events['type'].isin(defensive_types)]
    
    if len(defensive_attempts) < 10:
        return 0.5
    
    # Updated weights based on event importance
    weights = {
        'Clearance': 0.8,
        'Block': 1.0,
        'Interception': 1.2,
        'Duel': 1.0
    }
    
    def get_action_success(row: pd.Series) -> float:
        if row['type'] in ['Clearance', 'Block', 'Interception']:
            return weights[row['type']]
        elif row['type'] == 'Duel' and row['duel_outcome'] == 'Won':
            return weights['Duel']
        return 0
    
    success_scores = defensive_attempts.apply(get_action_success, axis=1)
    total_possible = sum(weights[t] for t in defensive_attempts['type'])
    
    return 0.5 if total_possible == 0 else success_scores.sum() / total_possible


In [4]:
def calculate_aerial_ability(events: pd.DataFrame, defender: str, team: str) -> float:
    """
    Calculate aerial ability with improved metrics and zone-based weighting
    Returns a score between 0-1
    """
    defender_events = events[events['player'] == defender]
    aerial_duels = defender_events[
        (defender_events['type'] == 'Duel') &
        (defender_events['duel_type'] == 'Aerial')
    ]
    
    if len(aerial_duels) < 15:
        return 0.5
    
    def get_duel_weight(location: List[float]) -> float:
        if not isinstance(location, list) or len(location) != 2:
            return 1.0
        x, _ = location
        if x <= 33:
            return 1.2  # Defensive third
        elif x <= 66:
            return 1.0  # Middle third
        return 0.8  # Attacking third
    
    weighted_duels = aerial_duels.apply(
        lambda x: get_duel_weight(x['location']) if x['duel_outcome'] == 'Won' else 0,
        axis=1
    )
    
    return weighted_duels.sum() / len(aerial_duels)

In [5]:
def calculate_one_v_one(events: pd.DataFrame, defender: str, team: str) -> float:
    """
    Calculate one-v-one defending ability with pressure consideration
    Returns a score between 0-1
    """
    defender_events = events[events['player'] == defender]
    tackles = defender_events[
        (defender_events['type'] == 'Duel') &
        (defender_events['duel_type'] == 'Tackle')
    ]
    
    if len(tackles) < 10:
        return 0.5
    
    successful_tackles = len(tackles[tackles['duel_outcome'] == 'Won'])
    tackle_success_rate = successful_tackles / len(tackles)
    
    recoveries = defender_events[defender_events['type'] == 'Ball Recovery']
    recovery_rate = len(recoveries) / max(len(defender_events) * 0.1, 1)
    
    # Consider pressure events
    pressure_events = len(defender_events[defender_events['type'] == 'Pressure'])
    pressure_factor = min(1.2, 1 + (pressure_events / 100))
    
    base_score = (tackle_success_rate * 0.75) + (min(recovery_rate, 1) * 0.25)
    return base_score * pressure_factor

In [6]:
def calculate_defensive_pressure(events: pd.DataFrame, defender: str, team: str) -> float:
    """
    Calculate defensive pressure effectiveness with improved success tracking
    Returns a score between 0-1
    """
    defender_events = events[events['player'] == defender]
    pressures = defender_events[defender_events['type'] == 'Pressure']
    
    if len(pressures) < 15:
        return 0.5
    
    def pressure_successful(events: pd.DataFrame, pressure_idx: int) -> float:
        if pressure_idx + 3 >= len(events):
            return 0
        next_events = events.iloc[pressure_idx + 1:pressure_idx + 4]
        if any(event['team'] == team for _, event in next_events.iterrows()):
            return 1.0
        return 0
    
    successful_pressures = sum(pressure_successful(events, i) for i in pressures.index)
    pressure_success_rate = successful_pressures / len(pressures)
    
    # Consider pressure location
    def get_pressure_weight(location: List[float]) -> float:
        if not isinstance(location, list) or len(location) != 2:
            return 1.0
        x, _ = location
        if x <= 33:
            return 1.2
        elif x <= 66:
            return 1.0
        return 0.8
    
    location_weights = pressures['location'].apply(get_pressure_weight)
    weighted_pressure_rate = (pressure_success_rate * location_weights).mean()
    
    return weighted_pressure_rate

In [7]:
def calculate_ball_progression(events: pd.DataFrame, defender: str, team: str) -> float:
    """
    Calculate ball progression ability with improved metrics
    Returns a score between 0-1
    """
    defender_events = events[events['player'] == defender]
    passes = defender_events[defender_events['type'] == 'Pass']
    
    if len(passes) < 20:
        return 0.5
    
    def calculate_progression_value(start_loc: List[float], end_loc: List[float]) -> float:
        if not (isinstance(start_loc, list) and isinstance(end_loc, list)):
            return 0
        start_x, _ = start_loc
        end_x, _ = end_loc
        progression = end_x - start_x
        
        if start_x <= 33:
            return progression / 15 if progression >= 15 else 0
        elif start_x <= 66:
            return progression / 10 if progression >= 10 else 0
        else:
            return progression / 5 if progression >= 5 else 0
    
    progressive_passes = passes.apply(
        lambda x: calculate_progression_value(x['location'], x['pass_end_location']),
        axis=1
    )
    
    # Consider pass completion rate
    completion_rate = len(passes[passes['pass_outcome'].isna()]) / len(passes)
    
    return (progressive_passes.mean() * 0.7 + completion_rate * 0.3)

In [8]:
def normalize_metric(series: pd.Series, min_val: float = 4, max_val: float = 10, 
                    minutes_played: pd.Series = None) -> Tuple[pd.Series, float]:
    """
    Normalize metrics with improved handling of edge cases and confidence intervals
    """
    if len(series) == 0:
        return pd.Series([]), 0
    
    if minutes_played is not None:
        min_minutes = 90  # One full match minimum
        reliability_factor = np.minimum(minutes_played / (4 * 90), 1)  # Scale up to 4 matches
        weights = reliability_factor
        series = series * weights
    
    if series.std() < 1e-10:
        return pd.Series([6.5] * len(series)), 0
    
    # Use robust bounds based on sample size
    if len(series) < 10:
        lower_bound = series.min() - (series.std() * 0.5)
        upper_bound = series.max() + (series.std() * 0.5)
    else:
        lower_bound = series.quantile(0.05)
        upper_bound = series.quantile(0.95)
    
    # Normalize and scale
    normalized = (series - lower_bound) / (upper_bound - lower_bound)
    normalized = np.clip(normalized, 0, 1)
    
    # Dynamic range based on sample reliability
    if minutes_played is not None:
        min_val_adj = min_val + (1 - reliability_factor.mean()) * 2
        max_val_adj = max_val - (1 - reliability_factor.mean()) * 2
    else:
        min_val_adj = min_val
        max_val_adj = max_val
    
    scaled = normalized * (max_val_adj - min_val_adj) + min_val_adj
    
    # Calculate confidence interval
    std_err = series.std() / np.sqrt(len(series))
    conf_interval = 1.96 * std_err
    
    return scaled.round(1), conf_interval

In [9]:
def analyze_defender_performance(competition_id: int, season_id: int) -> pd.DataFrame:
    """
    Analyze defender performance across multiple metrics with improved aggregation
    Returns a DataFrame with normalized scores and confidence intervals
    
    Parameters:
    competition_id (int): ID of the competition to analyze
    season_id (int): ID of the season to analyze
    
    Returns:
    pandas.DataFrame: Aggregated defender statistics with normalized scores
    """
    config = CompetitionConfig(competition_id, season_id)
    matches = sb.matches(competition_id=competition_id, season_id=season_id)
    all_defender_stats = []
    
    for _, match in matches.iterrows():
        try:
            events = sb.events(match_id=match['match_id'])
            defenders = events[events['position'].isin([
                'Right Center Back', 'Left Center Back', 'Center Back'
            ])]['player'].unique()
            
            for defender in defenders:
                defender_events = events[events['player'] == defender]
                if len(defender_events) == 0:
                    continue
                
                team = defender_events['team'].iloc[0]
                
                # Calculate minutes played including added time
                start_minute = defender_events['minute'].min()
                end_minute = defender_events['minute'].max()
                added_time = defender_events['second'].max() / 60 if end_minute >= 90 else 0
                minutes = end_minute - start_minute + added_time
                
                # Calculate all metrics
                metrics = {
                    'defender': defender,
                    'team': team,
                    'match_id': match['match_id'],
                    'minutes_played': minutes,
                    'box_defending': calculate_box_defending(events, defender, team),
                    'aerial_ability': calculate_aerial_ability(events, defender, team),
                    'one_v_one': calculate_one_v_one(events, defender, team),
                    'defensive_pressure': calculate_defensive_pressure(events, defender, team),
                    'ball_progression': calculate_ball_progression(events, defender, team)
                }
                
                all_defender_stats.append(metrics)
                
        except Exception as e:
            print(f"Error processing match {match['match_id']}: {str(e)}")
            continue
    
    # Create DataFrame
    df = pd.DataFrame(all_defender_stats)
    if len(df) == 0:
        return pd.DataFrame()
    
    # Group by defender and calculate weighted averages
    metrics = ['box_defending', 'aerial_ability', 'one_v_one', 
              'defensive_pressure', 'ball_progression']
    
    def weighted_avg(x):
        return np.average(x, weights=df.loc[x.index, 'minutes_played'])
    
    grouped_df = df.groupby(['defender', 'team']).agg({
        'minutes_played': 'sum',
        **{metric: weighted_avg for metric in metrics}
    }).reset_index()
    
    # Add matches played and full matches equivalent
    matches_played = df.groupby('defender')['match_id'].nunique()
    grouped_df['matches_played'] = grouped_df['defender'].map(matches_played)
    grouped_df['full_match_equivalent'] = (grouped_df['minutes_played'] / 90).round(1)
    
    # Normalize all metrics and add confidence intervals
    normalized_metrics = {}
    confidence_intervals = {}
    
    for metric in metrics:
        normalized, conf_interval = normalize_metric(
            grouped_df[metric],
            minutes_played=grouped_df['minutes_played']
        )
        normalized_metrics[metric] = normalized
        confidence_intervals[f'{metric}_ci'] = conf_interval
    
    # Add normalized metrics and confidence intervals to DataFrame
    for metric in metrics:
        grouped_df[metric] = normalized_metrics[metric]
        grouped_df[f'{metric}_ci'] = confidence_intervals[f'{metric}_ci']
    
    # Calculate composite score
    grouped_df['composite_score'] = grouped_df[metrics].mean(axis=1)
    
    # Sort by composite score
    grouped_df = grouped_df.sort_values('composite_score', ascending=False)
    
    # Round specific columns
    round_cols = metrics + [m + '_ci' for m in metrics] + ['composite_score', 'minutes_played']
    grouped_df[round_cols] = grouped_df[round_cols].round(2)
    
    # Set defender as index
    return grouped_df.set_index('defender')

In [ ]:


# Define leagues configuration
LEAGUES = [
    {"competition_id": 9, "season_ids": [281]},
    {"competition_id": 43, "season_ids": [106]},
    {"competition_id": 11, "season_ids": [90, 42, 4]},
    {"competition_id": 7, "season_ids": [235, 108]},
    {"competition_id": 2, "season_ids": [44]},
    {"competition_id": 12, "season_ids": [27]},
    {"competition_id": 55, "season_ids": [282]},
]

def create_session_with_retries() -> requests.Session:
    """
    Create a requests session with retry strategy
    """
    session = requests.Session()
    retry_strategy = Retry(
        total=5,  # number of retries
        backoff_factor=1,  # wait 1, 2, 4, 8, 16 seconds between retries
        status_forcelist=[500, 502, 503, 504, 404],
        allowed_methods=["HEAD", "GET", "OPTIONS"]
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

def fetch_with_retry(url: str, max_attempts: int = 3) -> dict:
    """
    Fetch data with retry logic and exponential backoff
    """
    session = create_session_with_retries()
    
    for attempt in range(max_attempts):
        try:
            response = session.get(url, timeout=30)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            if attempt == max_attempts - 1:
                raise
            wait_time = (2 ** attempt) + np.random.uniform(0, 1)
            print(f"Attempt {attempt + 1} failed. Waiting {wait_time:.2f} seconds before retry...")
            time.sleep(wait_time)

def analyze_competition_season(competition_id: int, season_id: int) -> pd.DataFrame:
    """
    Wrapper function to analyze a specific competition and season with error handling
    """
    try:
        print(f"\nAnalyzing competition {competition_id}, season {season_id}...")
        
        # Add delay between requests to avoid rate limiting
        time.sleep(2)
        
        result = analyze_defender_performance(competition_id, season_id)
        if result is not None and not result.empty:
            result['competition_id'] = competition_id
            result['season_id'] = season_id
            return result
        else:
            print(f"No data returned for competition {competition_id}, season {season_id}")
            return pd.DataFrame()
    except Exception as e:
        print(f"Error analyzing competition {competition_id}, season {season_id}: {str(e)}")
        return pd.DataFrame()







def main():
    all_results = []
    total_analyses = sum(len(league['season_ids']) for league in LEAGUES)
    
    print(f"Starting analysis of {total_analyses} competition-season combinations...")
    print("Note: Added delays between requests to avoid rate limiting")
    
    # Create progress bar
    with tqdm(total=total_analyses, desc="Analyzing competitions") as pbar:
        # Process each league and season
        for league in LEAGUES:
            competition_id = league['competition_id']
            for season_id in league['season_ids']:
                try:
                    # Add delay between competitions
                    time.sleep(3)
                    
                    result = analyze_competition_season(competition_id, season_id)
                    if not result.empty:
                        all_results.append(result)
                        print(f"Successfully analyzed competition {competition_id}, season {season_id}")
                except Exception as e:
                    print(f"Failed to analyze competition {competition_id}, season {season_id}: {str(e)}")
                finally:
                    pbar.update(1)
    
    if not all_results:
        print("No results were generated. Please check your internet connection and try again.")
        return
    
    # Combine all results
    combined_results = pd.concat(all_results)
    
    # Reset index to handle duplicates
    combined_results = combined_results.reset_index()
    
    # Handle duplicate players by keeping the record with most minutes played
    print("\nHandling duplicate players...")
    before_dedup = len(combined_results)
    
    # Sort by minutes_played in descending order and keep first occurrence of each player
    combined_results = combined_results.sort_values('minutes_played', ascending=False)
    combined_results = combined_results.drop_duplicates(subset=['defender'], keep='first')
    
    after_dedup = len(combined_results)
    if before_dedup != after_dedup:
        print(f"Removed {before_dedup - after_dedup} duplicate player entries")
        print("Kept entries with highest minutes played for each player")
    
    # Set index for final results
    combined_results.set_index(['defender'], inplace=True)
    
    # Sort by composite score
    combined_results = combined_results.sort_values('composite_score', ascending=False)
    
    # Save detailed results
    output_filename = f"defender_analysis_all_leagues_{time.strftime('%Y%m%d_%H%M%S')}.csv"
    combined_results.to_csv(output_filename)
    print(f"\nDetailed results saved to {output_filename}")
    
    # Create summary of top performers (now across all competitions)
    print("\nTop 10 Defenders Overall (based on highest minutes played entry):")
    print("-" * 80)
    top_10 = combined_results[['team', 'competition_id', 'season_id', 'minutes_played', 'composite_score']].head(10)
    print(top_10)
    
    # Save summary
    summary_filename = f"defender_analysis_summary_{time.strftime('%Y%m%d_%H%M%S')}.csv"
    top_10.to_csv(summary_filename)
    print(f"\nSummary of top performers saved to {summary_filename}")
    
    # Additional statistics
    print("\nAnalysis Statistics:")
    print(f"Total unique defenders analyzed: {len(combined_results)}")
    print(f"Number of competitions covered: {combined_results['competition_id'].nunique()}")
    print(f"Average minutes played per defender: {combined_results['minutes_played'].mean():.2f}")
    print(f"Average composite score: {combined_results['composite_score'].mean():.2f}")

if __name__ == "__main__":
    main()